# 04. 모델링 전 데이터 준비

**담당:** 정수진 · **Issue:** #65  
**대상:** `P1-CR1-CAL01` 소둔로 · `P1-CR1-WLD01` 용접 기록  
**분석 단위:** LOT·공정·시간 구간·센서별 관측  
**재구성일:** 2026-10-12

## 분석 목적

**분석 목적:** 앞선 근거를 시간 구간·센서 품질·운전 형태·작업 맥락·검토 플래그로 보존하고, 미래 정보를 쓰지 않는 특징과 시간 순서 평가 구성을 만든다. 실제 고장·더미 라벨이 없다는 점을 목표 설계에 반영한다. 모델 학습과 성능 주장은 수행하지 않는다.

네 파일의 연결: **01 시간 흐름 → 02 통계와 LOT 선정 → 03 가설 검증 → 04 모델링 준비**. 각 파일은 같은 8개 원본 CSV를 독립적으로 읽어 실행합니다. 분석 표는 노트북 출력에 남기며, 그림만 지정된 `figures/05_all_data_timeseries_jinny_04`에 저장합니다.

## 04 상세 목차 — 무엇을 보고 무엇을 확인할 것인가

| 순서 | 분석 항목 | 볼 자료와 핵심 질문 | 확인 결과와 연결 |
|---:|---|---|---|
| 1 | 검증 근거 재계산 | 원본·02와 같은 품질/상태/후보 규칙: 입력이 재현되는가? | 독립 실행·일관된 근거 |
| 2 | 실제 근거와 입력 대조 | 기록 누락 → 동일 강종 TC/OP → CP와 작업 시각 → 감지 전후·정비 후 미관측: 무엇이 실제로 비교되는가? | 유효 초·온도 차이·시간 간격·표본 수 → 센서별 NA/실제 입력/사후 맥락 판단 |
| 3 | 목표와 라벨 | 실제 고장·더미·정상 여부: 무엇을 정답으로 둘 수 있는가? | 미확정 라벨 NA·가능한 모델 목적 |
| 4 | 과거 정보 특징 | 현재 분·과거 이동 통계·강종: 예측 시점에 알 수 있는 값인가? | 시점별 특징·이용 가능 시각 |
| 5 | 학습·평가 분할 | 전체 LOT·달력 순서·강종 수: 미래 LOT이 학습에 들어가는가? | LOT 단위 시간 분할·지원 부족 강종 |
| 6 | 학습 기준값 | 학습 LOT만의 센서/강종 수준·참조 수: 평가 자료가 기준에 섞이는가? | 학습 전용 기준·미지원 강종 마스크 |
| 7 | 누출·연결 검증 | 미래 값 변조·분 경계·키·품질·구간 합: 결과가 과거 특징에 역류하는가? | 검증 통과/제약과 데이터 일관성 |
| 8 | 최종 활용값 | 구간·센서·품질·상태·검토 이유·특징·분할: 무엇을 남기는가? | 사용 가능 입력·평가 조건·추가 자료 필요사항 |

## 기본 설정

관측 시작·종료는 제공된 기록의 범위이며 소재의 실제 입·출측 시각이 아닙니다. TC의 ℃와 OP의 %는 기존 분석의 표시 가정을 유지합니다. 원본에 단위 사전·목표값·운전 모드가 없어 공학적 정상 한계는 정하지 않습니다. CP-Z4M은 정의와 단위가 확인되지 않아 원본 값을 그대로 표시합니다.

In [ ]:
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
# 이걸 기준으로 코드 작성할 것
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

DATA_DIR = os.path.join("..", "data")
FIG_DIR = os.path.join("..", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")

ENC = "utf-8-sig"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

## 자료 읽기와 연결

LOT 시작과 CR1 강종변경 시각의 정확 일치를 확인하고, CAL01 설비의 WO와 부품교체를 실제 키로 연결합니다. 온도 LOT과 용접 COIL 사이의 공통 소재 키는 없으므로 시간 겹침은 동시 관측의 근거로만 사용합니다. 수리의 마지막 날짜는 하루 전체를 포함하는 계획으로 읽습니다.

In [ ]:
NOTEBOOK_NO = "04"
from pathlib import Path
from IPython.display import display, Markdown, Image
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from scipy.signal import find_peaks

WLD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")
files = {"온도": TEMP_CSV, "조업": EVENT_CSV, "수리": REPAIR_CSV,
         "설비": EQP_CSV, "부품": PART_CSV, "정비": MAINT_CSV,
         "교체": PARTCHG_CSV, "용접": WLD_CSV}
frames = {name: pd.read_csv(path, encoding=ENC) for name, path in files.items()}
temp, event, repair, eqp = [frames[k] for k in ["온도", "조업", "수리", "설비"]]
part, maint, partchg, wld = [frames[k] for k in ["부품", "정비", "교체", "용접"]]
for frame, columns in [(temp, ["MEAS_DT"]), (event, ["EVT_DT"]),
                       (repair, ["STRT_DT", "END_DT"]),
                       (maint, ["DETC_DT", "STRT_DT", "END_DT"]), (wld, ["MEAS_DT"])]:
    for column in columns:
        frame[column] = pd.to_datetime(frame[column])
CAL_TAG, WLD_TAG = "P1-CR1-CAL01", "P1-CR1-WLD01"
YEAR_START, YEAR_END = pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01")
grade_evt = event.loc[event.PLANT_CD.eq("CR1") & event.EVT_TYPE.eq("강종변경")].sort_values("EVT_DT")
assert not grade_evt.EVT_DT.duplicated().any()
lots = temp.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), N_ROWS=("MEAS_DT", "size"))
lots = lots.merge(grade_evt[["EVT_DT", "AFT_VAL"]], left_on="START", right_on="EVT_DT",
                  how="left", validate="one_to_one").rename(columns={"AFT_VAL": "GRADE"})
lots = lots.sort_values("START").reset_index(drop=True)
assert lots.GRADE.notna().all()
lots["MINUTES"] = (lots.END - lots.START).dt.total_seconds() / 60
lot_meta = lots.set_index("LOT_NO")
cal_work = maint.loc[maint.EQP_TAG.eq(CAL_TAG)].sort_values("STRT_DT").copy()
assert cal_work.WO_NO.is_unique and cal_work.END_DT.ge(cal_work.STRT_DT).all()
cal_parts = (partchg.merge(cal_work[["WO_NO", "EQP_TAG", "MNT_TYPE", "DETC_DT", "STRT_DT", "END_DT", "REMARK"]],
                           on="WO_NO", validate="many_to_one")
             .merge(part, on="PART_CD", validate="many_to_one").sort_values("END_DT"))
plans = repair.loc[repair.PLANT_CD.eq("CR1")].sort_values("STRT_DT").copy()
plans["END_EXCL"] = plans.END_DT.dt.normalize() + pd.Timedelta(days=1)
coils = wld.groupby("COIL_ID", as_index=False).agg(
    START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), N_SEQ=("SEQ_NO", "size"),
    PWR_MIN=("PWR", "min"), PWR_MAX=("PWR", "max"), PWR_MEDIAN=("PWR", "median"))
coils["ZERO_ALL"] = coils.PWR_MIN.eq(0) & coils.PWR_MAX.eq(0)
coils = coils.sort_values("START")
TC = ["CLN-IN"] + [f"TC-Z{i}" for i in range(1, 10)] + ["TC-OUT1", "TC-OUT2"]
OP = [f"OP-Z{i}" for i in range(1, 9)]
SIGNALS = TC + OP + ["CP-Z4", "CP-Z4M"]
HEAT = [f"TC-Z{i}" for i in range(3, 7)]
PROCESS = {"CLN-IN": "세정", **{f"TC-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 10)},
           "TC-OUT1": "냉각", "TC-OUT2": "냉각",
           **{f"OP-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 9)},
           "CP-Z4": "가열·균열", "CP-Z4M": "가열·균열"}
LOT_COLOR = {240001: "#455a64", 240024: "#d55e00", 240037: "#0072b2", 240038: "#0072b2",
             240044: "#0072b2", 240058: "#a23b57", 240061: "#e69f00", 240069: "#7b2cbf",
             240091: "#009e73", 240108: "#0072b2", 240114: "#e69f00", 240117: "#7b2cbf",
             240121: "#009e73", 240133: "#7b2cbf"}
COL = {"LOT": "#087f9c", "GRADE": "#477eb5", "WLD": "#de920a", "BM": "#b52a49",
       "CBM": "#d05c76", "TBM": "#e38592", "PART": "#7540a8", "PLAN": "#b8e2f2",
       "WORK": "#9a6324", "DETECT": "#34495e", "FOCUS": "#c83745", "QUALITY": "#bfc8cf"}
DUMMY_PRIOR = {"SPHC": "사용자 후보 가정", "SPCC": "사용자 후보 가정", "SPCE": "사용자 후보 가정",
               "SPCEN": "사용자 후보 가정", "SPFH590": "사용자 제외 가정", "DP590": "사용자 제외 가정",
               "SGCC": "사용자 제외 가정", "SPCD": "추가 검토"}
lots["DUMMY_PRIOR"] = lots.GRADE.map(DUMMY_PRIOR)
context_rows = []
for r in lots.itertuples():
    tasks = cal_work.loc[(cal_work.STRT_DT < r.END) & (cal_work.END_DT > r.START)]
    detected = cal_work.loc[cal_work.DETC_DT.between(r.START, r.END)]
    wc = coils.loc[(coils.START <= r.END) & (coils.END >= r.START)]
    relationships = []
    for p in plans.itertuples():
        before = lots.loc[lots.END < p.STRT_DT].tail(1)
        after = lots.loc[lots.START >= p.END_EXCL].head(1)
        if len(before) and before.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 전 마지막 관측")
        if len(after) and after.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 후 첫 관측")
    context_rows.append({"LOT": int(r.LOT_NO), "GRADE": r.GRADE, "START": r.START, "END": r.END,
                         "MINUTES": r.MINUTES, "CAL_WORK": len(tasks), "CAL_DETECT": len(detected),
                         "WELD_COILS": len(wc), "PLAN_RELATION": " · ".join(relationships),
                         "DUMMY_PRIOR": DUMMY_PRIOR[r.GRADE]})
context = pd.DataFrame(context_rows).sort_values("START")
OUT_DIR = Path(FIG_DIR) / f"05_all_data_timeseries_jinny_{NOTEBOOK_NO}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
figure_checks = []
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)

def note(text):
    display(Markdown(text))

def table(frame, columns=None):
    shown = frame if columns is None else frame[columns]
    # 작은 CP-Z4M도 0으로 반올림하지 않고 과학 표기법으로 보여준다.
    with pd.option_context("display.float_format", lambda value: f"{value:.4g}"):
        display(shown)


## 그림 표시 규칙

짧은 제목, 공정과 컬럼이 보이는 소제목, 축의 대상·단위, 색 범례를 사용합니다. TC는 왼쪽, 해당 OP는 오른쪽에 놓고 CP-Z4와 CP-Z4M도 왼쪽·오른쪽으로 대조합니다. TC-Z9에는 OP가 제공되지 않아 가상의 쌍을 만들지 않습니다. 모든 그림은 저장 크기에서 글자 겹침·잘림과 범례의 그래프 침범을 검사합니다.

In [ ]:
rcParams.update({"font.size": 11.5, "axes.titlesize": 12.5, "axes.labelsize": 11.5,
                 "xtick.labelsize": 10, "ytick.labelsize": 10.5, "savefig.dpi": 160})

def unit(signal):
    if signal in TC:
        return "온도[℃]"
    if signal in OP:
        return "제어출력[%]"
    if signal == "CP-Z4M":
        return "CP-Z4M 기록값\n[정의·단위 미확인]"
    if signal == "CP-Z4":
        return "CP-Z4 기록값\n[단위 미확인]"
    return signal

def style(ax, xlabel="관측 시작 후 경과시간[분]"):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#a9bac7")
    ax.grid(color="#e4ebf0", linewidth=.6)
    ax.set_axisbelow(True)
    ax.set_xlabel(xlabel, labelpad=8)
    ax.tick_params(pad=5)

def header(fig, title, subtitle="", handles=(), ncol=4):
    fig.suptitle(title, x=.035, y=.975, ha="left", va="top",
                 fontsize=20, fontweight="bold", color="#17344d")
    if subtitle:
        fig.text(.035, .924, subtitle, ha="left", va="top", fontsize=10.8, color="#526d80")
    if handles:
        fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.031, .879),
                   ncol=ncol, frameon=False, fontsize=10.5, handlelength=2.4,
                   columnspacing=1.8, labelspacing=.65)

def lot_handles(ids):
    return [Line2D([0], [0], color=LOT_COLOR[int(k)], lw=2,
                   label=f"{int(k)} · {lot_meta.loc[int(k), 'GRADE']} · {lot_meta.loc[int(k), 'START']:%m/%d}") for k in ids]

def text_box(artist, renderer):
    patch = artist.get_bbox_patch() if hasattr(artist, "get_bbox_patch") else None
    return patch.get_window_extent(renderer) if patch is not None else artist.get_window_extent(renderer)

def finish(fig, name, svg=False):
    """저장할 크기에서 모든 글자와 범례의 겹침·잘림을 확인한다."""
    fig.set_dpi(160)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    artists = list(fig.texts) + list(fig.legends)
    for ax in fig.axes:
        artists += list(ax.texts)
        if not ax.axison:
            continue
        artists += [ax.title, ax._left_title, ax.xaxis.label, ax.yaxis.label,
                    ax.xaxis.offsetText, ax.yaxis.offsetText]
        for axis in [ax.xaxis, ax.yaxis]:
            low, high = sorted(axis.get_view_interval())
            for tick in axis.get_major_ticks():
                if low <= tick.get_loc() <= high:
                    artists += [tick.label1, tick.label2]
    unique = {}
    for artist in artists:
        if not artist.get_visible():
            continue
        if hasattr(artist, "get_text") and not artist.get_text():
            continue
        unique[id(artist)] = artist
    boxes = [(artist, text_box(artist, renderer)) for artist in unique.values()]
    for artist, box in boxes:
        if box.x0 < -.5 or box.y0 < -.5 or box.x1 > fig.bbox.x1 + .5 or box.y1 > fig.bbox.y1 + .5:
            raise RuntimeError(f"글자 잘림: {name}: {getattr(artist, 'get_text', lambda: '범례')()}")
    for i, (artist, box) in enumerate(boxes):
        for other, other_box in boxes[i + 1:]:
            if box.overlaps(other_box):
                raise RuntimeError(f"글자 겹침: {name}: {getattr(artist, 'get_text', lambda: '범례')()} / {getattr(other, 'get_text', lambda: '범례')()}")
    for legend in fig.legends:
        box = legend.get_window_extent(renderer)
        if any(box.overlaps(ax.get_window_extent(renderer)) for ax in fig.axes if ax.axison):
            raise RuntimeError(f"범례가 그래프를 가림: {name}")
    path = OUT_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160, facecolor="white")
    if svg:
        vector = path.with_suffix(".svg")
        fig.savefig(vector, facecolor="white")
        vector.write_text("\n".join(line.rstrip() for line in vector.read_text(encoding="utf-8").splitlines()) + "\n", encoding="utf-8")
    figure_checks.append({"FILE": name, "TITLE": fig._suptitle.get_text(), "N_AXES": len(fig.axes),
                          "TEXT_OVERLAP": "통과", "TEXT_BOUNDS": "통과", "LEGEND_BOUNDS": "통과"})
    display(Image(filename=str(path), width=1440))
    plt.close(fig)


## 1. 검증 근거 재계산

**목적:** 앞 파일의 메모리·이미지·결과 파일에 의존하지 않고 같은 원본과 규칙으로 근거를 재현한다. 02의 품질·상태·대조·CP 후보와 03의 원본 반복 확인을 다시 계산한다. 아래 사후 상태와 검토 플래그는 근거표이며 온라인 예측의 입력이 아니다.

In [ ]:
# 1초 기대 간격은 원본 시각 차이의 최빈값으로 확인한다. 결측은 보간하지 않는다.
raw, profile, quality, expected, states = {}, {}, {}, {}, {}
quality_rows, state_rows = [], []
STATE_COLOR = {"안정 참고": "#009e73", "초기 승온": "#e69f00", "종료 전환": "#d55e00"}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = temp.loc[temp.LOT_NO.eq(k)].sort_values("MEAS_DT").copy()
    assert not g.MEAS_DT.duplicated().any()
    gaps_s = g.MEAS_DT.diff().dt.total_seconds().dropna()
    assert gaps_s.mode().iloc[0] == 1 and gaps_s.ge(1).all()
    g["ELAPSED"] = (g.MEAS_DT - r.START).dt.total_seconds() / 60
    g["MINUTE"] = np.floor(g.ELAPSED).astype(int)
    grid = pd.RangeIndex(int(np.floor(r.MINUTES)) + 1, name="MINUTE")
    p = g.groupby("MINUTE")[SIGNALS].agg(["median", "min", "max", "count"]).reindex(grid)
    exp = pd.Series(np.minimum(60, int((r.END-r.START).total_seconds()) + 1 - grid.to_numpy()*60), index=grid)
    counts = g.groupby("MINUTE").size().reindex(grid, fill_value=0)
    q = p.xs("count", axis=1, level=1).fillna(0).div(exp, axis=0)
    raw[k], profile[k], quality[k], expected[k] = g, p, q, exp
    for s in SIGNALS:
        quality_rows.append({"LOT": k, "GRADE": r.GRADE, "SENSOR": s, "ROWS": len(g),
                             "EXPECTED_SECONDS": int(exp.sum()), "MISSING_TIMES_S": int(exp.sum()-len(g)),
                             "SENSOR_NAN": int(g[s].isna().sum()), "VALID_RATIO": g[s].count()/exp.sum(),
                             "LOW_90_WINDOWS": int(q[s].lt(.9).sum())})
    # 사후 형태 분류다. 실제 운전 모드나 온라인 모델 입력으로 사용하지 않는다.
    st = pd.Series("안정 참고", index=grid, dtype="object")
    early = g.loc[g.ELAPSED.lt(5), HEAT].median()
    later = g.loc[g.ELAPSED.between(80,100), HEAT].median()
    rise = later - early
    warm_end, tail_start = np.nan, np.nan
    if rise.notna().all() and rise.min() >= 20 and g.ELAPSED.between(80,100).shape[0] >= 600:
        near = pd.concat([(p[(s,"median")]-later[s]).abs().le(5 if s=="TC-Z3" else 2) for s in HEAT], axis=1).all(axis=1)
        slow = pd.concat([p[(s,"median")].diff(5).abs().le(2) for s in HEAT[1:]], axis=1).all(axis=1)
        run = (near & slow & grid.to_series().ge(5)).rolling(5, min_periods=5).sum().eq(5)
        warm_end = int(run.index[run][0])+1 if run.any() else len(grid)
        st.loc[grid < warm_end] = "초기 승온"
    middle = g.loc[g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6), HEAT].median()
    tail = g.loc[g.ELAPSED.ge(max(0,r.MINUTES-5)), HEAT].median()
    if (tail-middle).notna().all() and (tail-middle).max() <= -10:
        midmask = g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6)
        drop = pd.concat([(p[(s,"median")]-middle[s]).le(-10) for s in HEAT], axis=1).sum(axis=1).ge(3)
        cp_drop = (p[("CP-Z4","median")]-g.loc[midmask,"CP-Z4"].median()).le(-.05).rolling(3).sum().eq(3)
        op = ["OP-Z4","OP-Z5","OP-Z6"]
        op_drop = (p.loc[:,[(s,"median") for s in op]].median(axis=1)-g.loc[midmask,op].median().median()).le(-10).rolling(3).sum().eq(3)
        hit = (drop | cp_drop | op_drop) & grid.to_series().ge(r.MINUTES*.6)
        if hit.any():
            tail_start = int(hit.index[hit][0])
            st.loc[grid >= tail_start] = "종료 전환"
    states[k] = st
    state_rows.append({"LOT": k, "GRADE": r.GRADE, "RISE_MIN_C": rise.min(), "RISE_MAX_C": rise.max(),
                       "WARM_END_MIN": warm_end, "TAIL_START_MIN": tail_start,
                       "STABLE_WINDOWS": int(st.eq("안정 참고").sum())})
quality_summary = pd.DataFrame(quality_rows)
state_summary = pd.DataFrame(state_rows)
def runs(mask, minimum=1):
    """연속 True의 [시작, 끝 제외) 경과 분을 돌려준다."""
    a = np.flatnonzero(mask.to_numpy(bool))
    if not len(a):
        return []
    groups = np.split(a, np.flatnonzero(np.diff(a)>1)+1)
    return [(int(mask.index[z[0]]), int(mask.index[z[-1]])+1) for z in groups if len(z)>=minimum]


In [ ]:
# 같은 강종·안정 참고 상태·같은 경과 분에서 대상 LOT을 뺀 대조만 쓴다.
# 비교할 수 없는 후반 구간을 잘라내거나 대조 중앙값으로 채우지 않는다.
comparison, sensor_rows, episode_rows = {}, [], []
NONCYCLIC = [s for s in TC if s not in ["TC-Z9","TC-OUT1","TC-OUT2"]]
for grade, sub in lots.groupby("GRADE",sort=False):
    ids = sub.LOT_NO.astype(int).tolist()
    grid = pd.RangeIndex(max(len(profile[k]) for k in ids))
    for s in SIGNALS:
        values = pd.DataFrame({k:profile[k][(s,"median")].where(quality[k][s].ge(.9)&states[k].eq("안정 참고")).reindex(grid) for k in ids})
        for k in ids:
            peers = values.drop(columns=k)
            n = peers.count(axis=1)
            center = peers.median(axis=1)
            delta = (values[k]-center).where(n.ge(1))
            d = pd.DataFrame({"VALUE":values[k],"CENTER":center,"N_PEERS":n,"DELTA":delta})
            comparison[k,s] = d
            v = delta.dropna()
            sensor_rows.append({"LOT":k,"GRADE":grade,"SENSOR":s,"COMPARE_WINDOWS":len(v),
                                "MEDIAN_DELTA":v.median(),"Q05_DELTA":v.quantile(.05),"Q95_DELTA":v.quantile(.95),
                                "MAX_ABS_DELTA":v.abs().max(),"MIN_PEERS":int(n.loc[v.index].min()) if len(v) else 0})
            if s in NONCYCLIC:
                for a,b in runs(delta.abs().ge(3)&n.ge(2),3):
                    block = d.loc[a:b-1]
                    episode_rows.append({"LOT":k,"GRADE":grade,"SENSOR":s,"START_MIN":a,"END_MIN":b,
                                         "DURATION_MIN":b-a,"MAX_ABS_DELTA_C":block.DELTA.abs().max(),
                                         "MIN_PEERS":int(block.N_PEERS.min())})
sensor_summary = pd.DataFrame(sensor_rows)
episodes = pd.DataFrame(episode_rows,columns=["LOT","GRADE","SENSOR","START_MIN","END_MIN","DURATION_MIN","MAX_ABS_DELTA_C","MIN_PEERS"])
# CP는 단위가 확인되지 않았으므로 자체 안정 참고 중앙값과 기록값 차이로만 찾는다.
cp_rows, cp_flags = [], {}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = raw[k]
    ref = profile[k][("CP-Z4","median")].where(states[k].eq("안정 참고")&quality[k]["CP-Z4"].ge(.9)).median()
    hit = (g["CP-Z4"]-ref).abs().ge(.05)
    cp_flags[k] = pd.Series(profile[k].index.isin(g.loc[hit,"MINUTE"]),index=profile[k].index)
    if hit.any():
        j = (g["CP-Z4"]-ref).abs().idxmax()
        peak = g.loc[j]
        # 5초 초과 시각 공백에서 분리한다. 짧은 공백도 보간하지 않고 실제 기록 행수·최대 간격을 따로 남긴다.
        cluster = ((hit != hit.shift())|g.MEAS_DT.diff().dt.total_seconds().gt(5)).cumsum()
        pulse = g.loc[cluster.eq(cluster.loc[j])&hit]
        cp_rows.append({"LOT":k,"GRADE":r.GRADE,"BASE":ref,"PEAK_MIN":peak.ELAPSED,"PEAK_TIME":peak.MEAS_DT,
                        "PEAK_VALUE":peak["CP-Z4"],"DELTA":peak["CP-Z4"]-ref,
                        "START_MIN":pulse.ELAPSED.min(),"END_MIN":pulse.ELAPSED.max()+1/60,
                        "PULSE_WIDTH_MIN":(pulse.MEAS_DT.max()-pulse.MEAS_DT.min()).total_seconds()/60+1/60,
                        "MAX_RECORD_GAP_S":pulse.MEAS_DT.diff().dt.total_seconds().max(),
                        "RECORDED_SECONDS":len(pulse),"TOTAL_HIT_SECONDS":int(hit.sum()),
                        "TOTAL_HIT_MIN_WINDOWS":int(cp_flags[k].sum())})
cp_candidates = pd.DataFrame(cp_rows,columns=["LOT","GRADE","BASE","PEAK_MIN","PEAK_TIME","PEAK_VALUE","DELTA","START_MIN","END_MIN","PULSE_WIDTH_MIN","MAX_RECORD_GAP_S","RECORDED_SECONDS","TOTAL_HIT_SECONDS","TOTAL_HIT_MIN_WINDOWS"])


In [ ]:
raw_delta,temperature_rows={},[]
def seconds_series(k,s):
    g=raw[k];ix=np.rint(g.ELAPSED.to_numpy()*60).astype(int)
    grid=pd.RangeIndex(int(np.floor(lot_meta.loc[k,"MINUTES"]*60))+1)
    y=pd.Series(g[s].to_numpy(),index=ix).reindex(grid)
    minute=grid.to_numpy()//60
    valid=states[k].reindex(minute).to_numpy()=="안정 참고"
    valid &= quality[k][s].reindex(minute).to_numpy()>=.9
    return y.where(valid)
for e in episodes.itertuples():
    key=e.LOT,e.SENSOR
    if key not in raw_delta:
        ids=lots.loc[lots.GRADE.eq(e.GRADE),"LOT_NO"].astype(int).tolist()
        target=seconds_series(e.LOT,e.SENSOR)
        peers=pd.DataFrame({k:seconds_series(k,e.SENSOR).reindex(target.index) for k in ids if k!=e.LOT})
        n=peers.count(axis=1)
        d=(target-peers.median(axis=1)).where(n.ge(2))
        raw_delta[key]=pd.DataFrame({"DELTA":d,"N_PEERS":n})
    block=raw_delta[key].loc[int(e.START_MIN*60):int(e.END_MIN*60)-1]
    v=block.DELTA.dropna();hit=block.DELTA.abs().ge(3)
    run=runs(hit);maxrun=max([b-a for a,b in run],default=0)
    ratio=hit.sum()/len(v) if len(v) else np.nan
    temperature_rows.append({"EPISODE_ID":e.Index+1,"LOT":e.LOT,"GRADE":e.GRADE,"SENSOR":e.SENSOR,
                             "START_MIN":e.START_MIN,"END_MIN":e.END_MIN,"DURATION_MIN":e.DURATION_MIN,
                             "RAW_COMPARE_SECONDS":len(v),"RAW_EXCEED_SECONDS":int(hit.sum()),"RAW_EXCEED_RATIO":ratio,
                             "MAX_CONTIGUOUS_RECORDED_S":maxrun,"RAW_MAX_ABS_DELTA_C":v.abs().max(),
                             "VERDICT":"원본 반복 확인" if len(v)>=60 and ratio>=.5 else "비율 부족·보류"})
temperature_validation=pd.DataFrame(temperature_rows)
comparison_evidence=temperature_validation[["LOT","SENSOR","START_MIN","END_MIN","RAW_COMPARE_SECONDS","RAW_EXCEED_SECONDS","RAW_EXCEED_RATIO","RAW_MAX_ABS_DELTA_C"]].copy()
comparison_evidence["RAW_EXCEED_RATIO"]*=100
table(comparison_evidence.rename(columns={"LOT":"LOT","SENSOR":"센서","START_MIN":"경과 시작[분]","END_MIN":"끝 제외[분]",
                                        "RAW_COMPARE_SECONDS":"실제 비교 초","RAW_EXCEED_SECONDS":"차이 ≥3℃ 초",
                                        "RAW_EXCEED_RATIO":"차이 ≥3℃ 비율[%]","RAW_MAX_ABS_DELTA_C":"최대 절대 차이[℃]"}))
window_rows=[]
for k,sub in episodes.groupby("LOT"):
    current=None
    for e in sub.sort_values(["START_MIN","END_MIN"]).itertuples():
        if current is None:
            current={"LOT":k,"START_MIN":e.START_MIN,"END_MIN":e.END_MIN,"SENSORS":{e.SENSOR}}
        elif e.START_MIN<=current["END_MIN"]+5:
            current["END_MIN"]=max(current["END_MIN"],e.END_MIN);current["SENSORS"].add(e.SENSOR)
        else:
            window_rows.append(current);current={"LOT":k,"START_MIN":e.START_MIN,"END_MIN":e.END_MIN,"SENSORS":{e.SENSOR}}
    if current is not None:window_rows.append(current)
temperature_windows=pd.DataFrame(window_rows)

note(f"**수치 대조 결과:** {len(temperature_validation)}개 센서 후보 중 {(temperature_validation.VERDICT=='원본 반복 확인').sum()}개 구간은 실제 비교 초의 절반 이상에서 3℃ 차이가 있다. 같은 수치 조건을 충족하지 않는 구간은 {temperature_validation.VERDICT.ne('원본 반복 확인').sum()}개다. 총 {len(temperature_windows)}개의 확대창을 원본·해당 OP·개별 후보로 03에서 전수 확인했다. 여기서는 그 확인 조건을 재현하여 구간별 근거에 사용한다.")

## 2. 실제 근거와 입력 처리 대조

**목적:** 플래그가 몇 개 있는지 대신, 같은 시간의 실제 센서값·대조값·유효 초·작업 시각을 보여준다. 각 근거로 어떤 현재 입력을 남기고 어떤 값은 사후 맥락 또는 미확정으로 관리해야 하는지 결정한다.

| 대조 순서 | 볼 자료와 질문 | 입력 준비에 연결할 판단 |
|---|---|---|
| 기록 누락·센서 NaN | 240044 전체→초기 원본, TC-Z3·OP-Z3의 각 분 유효 초: 같은 시간에 무엇이 실제로 기록됐는가? | 실제 값과 결측 구분; 센서별 54/60초 입력 마스크 |
| 같은 강종 온도·출력 | 240061 SPHC 전체→131–137분 원본, 대조 2 LOT: 얼마나 다른가? | 차이 크기·실제 비교 초·OP 대조; 후속 LOT 대조는 사후 해석에만 사용 |
| 실제 작업과 CP | 240117 전체→펄스 원본, TC-Z4·OP-Z4·CP-Z4·CP-Z4M: 작업 종료와 언제·얼마나 다른가? | 실제 신호·작업 맥락을 독립 보존; CP 정의 확인 전 보조 특징 |
| 감지 전후·정비 이후 | 240133 전체→감지 양쪽 20분, 작업·교체 WO 완료: 회복을 관측할 자료가 있는가? | 실제 전후 신호·표본 수; 정비 후 미관측이면 효과 라벨 NA |
| 최종 근거→입력 | 위 실제 수치와 활용 판단: 무엇을 입력·사후 검토·미확정으로 남기는가? | 값과 마스크·시간 정보의 처리 규칙 |

전체 분당 구간표와 센서별 유효율은 아래에서 재계산하며 내부 근거를 덮어쓰지 않는다. 시각 누락을 보간하지 않고, 교체의 정확한 실측 시각은 없어 WO 완료 맥락으로 표시한다.

54/60초(90%)와 3℃는 현재 분석에 적용한 입력·탐색 규칙이다. 공학적 정상/고장 한계나 모델 성능이 검증된 문턱을 뜻하지 않으며, 모델 선택 단계에서 학습 기간의 별도 시간 검증으로 확정해야 한다.

In [ ]:
# 사후 검토용 다차원 구간표: 어느 플래그도 다른 플래그를 덮어쓰지 않는다.
minute_rows=[]
for r in lots.itertuples():
    k=int(r.LOT_NO);p=profile[k];minutes=p.index.to_numpy()
    lo=r.START+pd.to_timedelta(minutes,unit="min")
    hi=pd.DatetimeIndex([min(t+pd.Timedelta(minutes=1),r.END+pd.Timedelta(seconds=1)) for t in lo])
    frame=pd.DataFrame({"LOT":k,"GRADE":r.GRADE,"MINUTE":minutes,"START":lo,"END_EXCL":hi,"REVIEW_STATE":states[k].to_numpy(),
                        "QUALITY_MIN_RATIO":quality[k].min(axis=1).to_numpy(),"LOW_QUALITY_ANY":quality[k].lt(.9).any(axis=1).to_numpy(),
                        "CP_REVIEW":cp_flags[k].to_numpy(),"TEMP_REVIEW":False,"RAW_TEMP_VERIFIED":False,
                        "CAL_WORK":False,"CAL_DETECT":False,"PART_WO_COMPLETED":False,"WELD_COILS":0})
    for t in cal_work.itertuples():
        frame.loc[(lo<t.END_DT)&(hi>t.STRT_DT),"CAL_WORK"]=True
        frame.loc[(lo<=t.DETC_DT)&(hi>t.DETC_DT),"CAL_DETECT"]=True
        if t.WO_NO in set(cal_parts.WO_NO):
            frame.loc[(lo<=t.END_DT)&(hi>t.END_DT),"PART_WO_COMPLETED"]=True
    for z in coils.loc[coils.START.le(r.END)&coils.END.ge(r.START)].itertuples():
        frame.loc[(lo<=z.END)&(hi>z.START),"WELD_COILS"]+=1
    for e in episodes.loc[episodes.LOT.eq(k)].itertuples():
        hit=frame.MINUTE.ge(e.START_MIN)&frame.MINUTE.lt(e.END_MIN)
        frame.loc[hit,"TEMP_REVIEW"]=True
    for e in temperature_validation.loc[temperature_validation.LOT.eq(k)&temperature_validation.VERDICT.eq("원본 반복 확인")].itertuples():
        hit=frame.MINUTE.ge(e.START_MIN)&frame.MINUTE.lt(e.END_MIN)
        frame.loc[hit,"RAW_TEMP_VERIFIED"]=True
    frame["REVIEW_INTERVAL_SECONDS"]=(frame.END_EXCL-frame.START).dt.total_seconds()
    minute_rows.append(frame)
minute_context=pd.concat(minute_rows,ignore_index=True).sort_values(["START","LOT"]).reset_index(drop=True)
for name in ["FAULT_LABEL","DUMMY_LABEL","NORMAL_LABEL"]:
    minute_context[name]=pd.Series(pd.NA,index=minute_context.index,dtype="boolean")
# 센서별 품질은 별도의 긴 표에 보존한다. 일부 센서 NaN으로 모든 센서를 지우지 않는다.
sensor_quality=pd.concat([quality[k].rename_axis("MINUTE").reset_index().melt(id_vars="MINUTE",var_name="SENSOR",value_name="VALID_RATIO").assign(LOT=k)
                          for k in lots.LOT_NO.astype(int)],ignore_index=True)
sensor_quality["LOW_QUALITY"]=sensor_quality.VALID_RATIO.lt(.9)
assert not minute_context.duplicated(["LOT","MINUTE"]).any()
assert not sensor_quality.duplicated(["LOT","MINUTE","SENSOR"]).any()


In [ ]:
flag_summary=minute_context.groupby(["LOT","GRADE"],as_index=False).agg(
    WINDOWS=("MINUTE","size"),REVIEW_SECONDS=("REVIEW_INTERVAL_SECONDS","sum"),
    LOW_QUALITY_ANY=("LOW_QUALITY_ANY","sum"),TEMP_REVIEW=("TEMP_REVIEW","sum"),RAW_TEMP_VERIFIED=("RAW_TEMP_VERIFIED","sum"),
    CP_REVIEW=("CP_REVIEW","sum"),WORK_WINDOWS=("CAL_WORK","sum"),DETECT_WINDOWS=("CAL_DETECT","sum"))


# 실제 곡선과 초 단위 공백을 표시한다. 요약용 플래그 개수를 그림으로 비교하지 않는다.
def evidence_trace(ax,k,s,lo=0,hi=None,detail=False,color=None):
    meta=lot_meta.loc[k]
    hi=meta.MINUTES if hi is None else min(hi,meta.MINUTES)
    if detail:
        ix=pd.RangeIndex(int(lo*60),int(hi*60)+1)
        times=meta.START+pd.to_timedelta(ix,unit="s")
        y=raw[k].set_index("MEAS_DT")[s].reindex(times).to_numpy()
        x=ix.to_numpy()/60
    else:
        p=profile[k].loc[lambda d:d.index.to_series().between(lo,hi)]
        x=p.index.to_numpy()
        y=p[(s,"median")].to_numpy()
    ax.plot(x,y,color=color or LOT_COLOR[k],lw=1.7)
    ax.set_ylabel(unit(s),labelpad=10)
    ax.set_title(f"{PROCESS[s]} [{s}]",loc="left",pad=12)
    style(ax)
    ax.xaxis.set_major_locator(MaxNLocator(5))
    ax.yaxis.set_major_locator(MaxNLocator(4))
    if s=="CP-Z4M":
        ax.ticklabel_format(axis="y",style="sci",scilimits=(0,0))

def exact_work_bands(ax,k,lo=0,hi=None):
    meta=lot_meta.loc[k]
    hi=meta.MINUTES if hi is None else hi
    tasks=cal_work.loc[cal_work.STRT_DT.lt(meta.START+pd.Timedelta(minutes=hi))&
                       cal_work.END_DT.gt(meta.START+pd.Timedelta(minutes=lo))]
    for t in tasks.itertuples():
        a=(t.STRT_DT-meta.START).total_seconds()/60
        b=(t.END_DT-meta.START).total_seconds()/60
        row=0 if t.MNT_TYPE=="TBM" else .065
        ax.axvspan(a,b,ymin=row,ymax=row+.055,color=COL[t.MNT_TYPE],alpha=.8,lw=0)
        ax.axvline(b,color=COL[t.MNT_TYPE],ls=":",lw=1)
    return tasks


note("**그림 목적 — 기록 누락과 입력 처리:** 240044의 전체 TC-Z3·OP-Z3에서 초기 0–5분을 표시한 뒤 원본으로 확대한다. 모든 센서의 시각 누락과 특정 센서의 NaN을 구분하고, 닫힌 60초 창에 실제 유효 기록이 54초 이상 있을 때만 현재 입력을 남기는 규칙을 확인한다.")
k=240044;meta=lot_meta.loc[k]
grid=pd.RangeIndex(300)
times=meta.START+pd.to_timedelta(grid,unit="s")
detail=raw[k].set_index("MEAS_DT").reindex(times)
present=times.isin(raw[k].MEAS_DT)
missing_clock=pd.Series(~present,index=grid)
sensor_missing=pd.Series(present&detail["OP-Z3"].isna().to_numpy(),index=grid)
first5=raw[k].loc[raw[k].ELAPSED.lt(5)].groupby("MINUTE")[["TC-Z3","OP-Z3"]].agg(["count","median"]).reindex(range(5))
input_evidence=[]
fig,axs=plt.subplots(2,2,figsize=(24,11))
fig.subplots_adjust(left=.075,right=.97,top=.76,bottom=.23,wspace=.24,hspace=.76)
for j,s in enumerate(["TC-Z3","OP-Z3"]):
    evidence_trace(axs[0,j],k,s)
    axs[0,j].axvspan(0,5,color="#bad8ee",alpha=.4,lw=0)
    axs[0,j].set_xlim(0,meta.MINUTES)
    axs[0,j].set_title(f"전체 {meta.MINUTES:.2f}분 · 가열·균열 [{s}]",loc="left",pad=12)
    ax=axs[1,j]
    evidence_trace(ax,k,s,0,5,detail=True)
    for a,b in runs(missing_clock):
        ax.axvspan(a/60,b/60,color="#bfc8cf",alpha=.8,lw=0)
    if s=="OP-Z3":
        for a,b in runs(sensor_missing):
            ax.axvspan(a/60,b/60,color="#e7a26f",alpha=.55,lw=0)
    for minute in range(5):
        n=int(first5.loc[minute,(s,"count")])
        use=n>=54
        if use:
            ax.scatter(minute+1,first5.loc[minute,(s,"median")],marker="o",color="#17344d",s=42,zorder=6)
        input_evidence.append({"LOT":k,"센서":s,"경과 분":minute,"예측 시점[경과 분]":minute+1,
                               "기대 초":60,"실제 유효 초":n,"유효율[%]":n/60*100,
                               "현재 입력":first5.loc[minute,(s,"median")] if use else np.nan})
    ax.set_xlim(0,5.05)
    ax.set_title(f"초기 0–5분 원본 · 가열·균열 [{s}]",loc="left",pad=12)
header(fig,"기록 누락·입력 대조",
       f"{k} · {meta.GRADE} · {meta.START:%m/%d %H:%M}–{meta.END:%H:%M:%S}; 위: 전체 1분 중앙값 / 아래: 초기 원본 초",
       [Line2D([0],[0],color=LOT_COLOR[k],lw=2,label=f"{k} · {meta.GRADE} 실제 기록"),
        Patch(facecolor="#bad8ee",label="아래에서 확대: 0–5분"),
        Patch(facecolor="#bfc8cf",label="시각 기록 없음: 모든 센서"),
        Patch(facecolor="#e7a26f",label="시각은 있으나 OP-Z3 값 NaN"),
        Line2D([0],[0],marker="o",ls="None",color="#17344d",label="54/60초 이상: 닫힌 1분 입력")],ncol=3)
valid_tc=first5[("TC-Z3","count")].astype(int).tolist()
valid_op=first5[("OP-Z3","count")].astype(int).tolist()
nan_times=times[sensor_missing.to_numpy()]
fig.text(.075,.160,f"같은 초기 5분의 유효 초: TC-Z3 {valid_tc} / OP-Z3 {valid_op} (각 분 기대 60초).",fontsize=13,color="#17344d",fontweight="bold")
fig.text(.075,.113,f"OP-Z3 NaN {int(sensor_missing.sum())}행: {nan_times.min():%H:%M:%S}–{nan_times.max():%H:%M:%S}; 두 시각 사이에 실제 기록 공백도 있어 연속 결측 초와 구분한다.",
         fontsize=12,color="#526d80")
fig.text(.075,.063,"입력 결정: 초기 TC-Z3 5개 창은 유지, OP-Z3 0·1분 창의 현재값은 NA. 일부 센서 결측으로 LOT 전체를 삭제하지 않는다.",
         fontsize=13,color="#17344d",fontweight="bold")
finish(fig,"01_구간근거/01_기록누락_입력대조.png")
input_evidence=pd.DataFrame(input_evidence);table(input_evidence)
note(f"**그래프 결과:** 초기 TC-Z3는 각 분 {min(valid_tc)}–{max(valid_tc)}초가 유효하여 현재 입력을 남길 수 있다. OP-Z3는 0·1분에서 각각 {valid_op[0]}·{valid_op[1]}초로 54초 기준에 미달해 해당 현재값만 NA로 둔다. 실제 관측 값의 변화와 기록 부재는 구분되며, 이 입력 마스크는 뒤의 features에서 같은 60초 분모로 검증한다.")


### 같은 강종의 실제 온도·출력 대조

**목적:** 승온 뒤 240061의 TC-Z3 차이가 얼마나 크고 얼마나 많은 실제 초에서 확인되는지 두 개 대조 LOT과 비교한다. TC와 OP를 함께 읽는다.

In [ ]:

note("**그림 목적 — 같은 강종 온도 대조:** 240061의 승온 뒤 TC-Z3 차이를 SPHC 다른 LOT과 같은 경과시간에서 비교한다. 전체에서 131–137분을 표시하고 원본으로 확대한다. 해당 초에 유효한 대조 2개 LOT을 확보한 경우의 차이와 OP-Z3를 함께 읽는다. 이 대조에는 후속 시기의 LOT이 포함돼 사후 검토용이며 온라인 입력으로 쓰지 않는다.")
k=240061
e=temperature_validation.loc[temperature_validation.LOT.eq(k)&temperature_validation.SENSOR.eq("TC-Z3")].sort_values("RAW_MAX_ABS_DELTA_C",ascending=False).iloc[0]
a,b=int(e.START_MIN),int(e.END_MIN)
lo,hi=max(0,a-4),b+4
peer_ids=[int(z) for z in lots.loc[lots.GRADE.eq("SPHC"),"LOT_NO"] if z!=k and lot_meta.loc[z,"MINUTES"]>=hi]
ix=pd.RangeIndex(lo*60,hi*60)
target=seconds_series(k,"TC-Z3").reindex(ix)
peers=pd.DataFrame({z:seconds_series(z,"TC-Z3").reindex(ix) for z in peer_ids})
center=peers.median(axis=1)
delta=(target-center).where(peers.count(axis=1).ge(2))
candidate_records=raw[k].loc[raw[k].ELAPSED.ge(a)&raw[k].ELAPSED.lt(b)]
op_zero_rows=int(candidate_records["OP-Z3"].eq(0).sum())
evidence_tc_op=[]
for s in ["TC-Z3","OP-Z3"]:
    tv=seconds_series(k,s)
    refs=pd.DataFrame({z:seconds_series(z,s).reindex(tv.index) for z in peer_ids})
    d=(tv-refs.median(axis=1)).where(refs.count(axis=1).ge(2)).loc[a*60:b*60-1].dropna()
    evidence_tc_op.append({"LOT":k,"센서":s,"경과 시작[분]":a,"끝 제외[분]":b,
                           "대조 LOT":", ".join(map(str,peer_ids)),"비교 가능 초":len(d),
                           "대조 중앙값과 차이 중앙값":d.median(),"최대 절대 차이":d.abs().max(),
                           "TC 3℃ 이상 초":int(d.abs().ge(3).sum()) if s=="TC-Z3" else np.nan})
fig,axs=plt.subplots(2,2,figsize=(24,11))
fig.subplots_adjust(left=.075,right=.97,top=.76,bottom=.24,wspace=.25,hspace=.76)
ids=lots.loc[lots.GRADE.eq("SPHC"),"LOT_NO"].astype(int).tolist()
for j,s in enumerate(["TC-Z3","OP-Z3"]):
    for z in ids:evidence_trace(axs[0,j],z,s)
    axs[0,j].axvspan(a,b,color="#c83745",alpha=.15,lw=0)
    axs[0,j].set_xlim(0,max(lot_meta.loc[z,"MINUTES"] for z in ids))
    axs[0,j].set_title(f"SPHC 전체 관측 · 가열·균열 [{s}]",loc="left",pad=12)
    ax=axs[1,j]
    for z in [k]+peer_ids:evidence_trace(ax,z,s,lo,hi,detail=True)
    ax.axvspan(a,b,color="#c83745",alpha=.10,lw=0)
    ax.set_xlim(lo,hi)
    ax.set_title(f"원본 {lo}–{hi}분 · 가열·균열 [{s}]",loc="left",pad=12)
hit=delta.abs().ge(3)&ix.to_series().ge(a*60)&ix.to_series().lt(b*60)
axs[1,0].fill_between(ix.to_numpy()/60,target.to_numpy(),center.to_numpy(),where=hit.to_numpy(),
                      color="#c83745",alpha=.3,lw=0)
header(fig,"동일 강종 온도·출력 대조",
       f"{k} SPHC · {a}–{b}분: 분당 요약의 차이가 원본 초에도 있는가? TC-Z3 왼쪽 / OP-Z3 오른쪽",
       lot_handles(ids)+[Patch(facecolor="#c83745",alpha=.3,label="TC 차이 ≥3℃인 실제 비교 초")],ncol=5)
fig.text(.075,.164,f"{a}–{b}분: 실제 비교 {int(e.RAW_COMPARE_SECONDS)}초 모두 |TC 차이|≥3℃, 최대 {e.RAW_MAX_ABS_DELTA_C:.3f}℃ / OP-Z3=0: {op_zero_rows}/{len(candidate_records)}행.",
         fontsize=13.5,color="#17344d",fontweight="bold")
fig.text(.075,.118,f"대조 {peer_ids} · 기록 공백을 채우지 않음. 6분의 시간 폭과 실제 비교 {int(e.RAW_COMPARE_SECONDS)}초는 다르다.",
         fontsize=12,color="#526d80")
fig.text(.075,.073,"입력 결정: 실제 TC·OP 값과 과거 특징은 유지하고 이 구간을 검토 대상으로 남긴다. 3℃는 탐색 기준이며 실제 고장 문턱이 아니다.",
         fontsize=12.6,color="#17344d",fontweight="bold")
fig.text(.075,.032,"후속 LOT을 포함한 이 대조 차이는 온라인 특징에서 제외한다. 모델 참고 차이는 6절의 학습 LOT 전용 기준으로 별도 계산한다.",
         fontsize=11.8,color="#526d80")
finish(fig,"01_구간근거/02_동일강종_온도출력_대조.png")
evidence_tc_op=pd.DataFrame(evidence_tc_op);table(evidence_tc_op)
note(f"**그래프 결과:** {k} {a}–{b}분 TC-Z3의 실제 비교 {int(e.RAW_COMPARE_SECONDS)}초에서 차이가 반복되고 최대 {e.RAW_MAX_ABS_DELTA_C:.3f}℃다. 같은 6분의 OP-Z3 원본 {len(candidate_records)}행 중 {op_zero_rows}행은 0이다. TC 상승과 출력 억제가 같이 나타나므로 목표값·운전 모드·제어 반응의 선후관계를 추가 확인한다. OP 차이 중앙값과 표본 수는 위 표에서 별도 확인했다. 온도 차이의 존재는 확인했지만 목표값·속도·두께·고장 정답이 없어 원인이나 고장으로 확정하지 않는다. 실제 값은 온라인 특징에, 이 비교 근거는 사후 해석에 남긴다.")


### 실제 정비와 CP·동시 신호 대조

**목적:** 240117의 CP 펄스를 TC·OP·CP-Z4M과 연결하고 실제 작업 종료와 시간 간격을 확인한다. CP의 주 입력 사용 여부를 판단한다.

In [ ]:

note("**그림 목적 — 정비 시각과 CP 대조:** 240117 전체 TC-Z4·OP-Z4·CP-Z4·CP-Z4M을 실제 작업 시각과 연결하고 CBM 종료 뒤 CP 펄스를 확대한다. 신호의 변화량과 작업과의 시간 간격을 구분해 CP를 주 입력으로 사용할 근거가 있는지 확인한다.")
k=240117;meta=lot_meta.loc[k]
cp=cp_candidates.loc[cp_candidates.LOT.eq(k)].iloc[0]
tasks=cal_work.loc[cal_work.STRT_DT.lt(meta.END)&cal_work.END_DT.gt(meta.START)].sort_values("STRT_DT")
cbm=tasks.loc[tasks.MNT_TYPE.eq("CBM")].iloc[0]
work_intervals=sorted((max(z.STRT_DT,meta.START),min(z.END_DT,meta.END)) for z in tasks.itertuples())
merged=[]
for begin,end in work_intervals:
    if merged and begin<=merged[-1][1]:
        merged[-1]=(merged[-1][0],max(end,merged[-1][1]))
    else:
        merged.append((begin,end))
work_union=sum((end-begin).total_seconds()/60 for begin,end in merged)
work_sum=sum((end-begin).total_seconds()/60 for begin,end in work_intervals)
shared_work=work_sum-work_union
tbm_duration=tasks.loc[tasks.MNT_TYPE.eq("TBM")].apply(lambda z:(z.END_DT-z.STRT_DT).total_seconds()/60,axis=1).sum()
cbm_duration=tasks.loc[tasks.MNT_TYPE.eq("CBM")].apply(lambda z:(z.END_DT-z.STRT_DT).total_seconds()/60,axis=1).sum()
cbm_end=(cbm.END_DT-meta.START).total_seconds()/60
peak_gap=(cp.PEAK_TIME-cbm.END_DT).total_seconds()/60
lo,hi=cbm_end-6,cp.END_MIN+10
pulse_effects=[]
for s in ["CP-Z4","CP-Z4M","TC-Z4","OP-Z4"]:
    g=raw[k]
    before=g.loc[g.ELAPSED.ge(cp.START_MIN-5)&g.ELAPSED.lt(cp.START_MIN),s].dropna()
    during=g.loc[g.ELAPSED.ge(cp.START_MIN)&g.ELAPSED.lt(cp.END_MIN),s].dropna()
    pulse_effects.append({"LOT":k,"센서":s,"펄스 전5분 중앙값":before.median(),"펄스 중앙값":during.median(),
                         "펄스-전5분":during.median()-before.median(),
                         "전5분 유효 행":len(before),"펄스 유효 행":len(during)})
pulse_effects=pd.DataFrame(pulse_effects)
fig,axs=plt.subplots(3,2,figsize=(24,14))
fig.subplots_adjust(left=.075,right=.97,top=.78,bottom=.18,wspace=.23,hspace=.76)
signals=[["TC-Z4","OP-Z4"],["CP-Z4","CP-Z4M"],["CP-Z4","CP-Z4M"]]
for ri,row in enumerate(signals):
    for j,s in enumerate(row):
        ax=axs[ri,j]
        evidence_trace(ax,k,s,lo if ri==2 else 0,hi if ri==2 else None,detail=ri==2)
        exact_work_bands(ax,k,lo if ri==2 else 0,hi if ri==2 else None)
        ax.axvspan(cp.START_MIN,cp.END_MIN,color="#b04482",alpha=.2,lw=0)
        if ri<2:
            warm=state_summary.set_index("LOT").loc[k,"WARM_END_MIN"]
            ax.axvspan(0,warm,color="#f5d98c",alpha=.2,lw=0)
            ax.set_xlim(0,meta.MINUTES)
            ax.set_title(f"전체 {meta.MINUTES:.2f}분 · {PROCESS[s]} [{s}]",loc="left",pad=12)
        else:
            ax.set_xlim(lo,hi)
            ax.set_title(f"CBM 종료 전후 원본 · {PROCESS[s]} [{s}]",loc="left",pad=12)
        if s=="CP-Z4":
            ax.axhline(cp.BASE,color="#526d80",ls="--",lw=1.1)
header(fig,"정비 시각·CP 기록 대조",
       f"{k} SPCD · {meta.START:%m/%d %H:%M}–{meta.END:%H:%M:%S}; 전체 → CP 펄스 원본; TC 왼쪽·OP 오른쪽, CP-Z4 왼쪽·CP-Z4M 오른쪽",
       [Line2D([0],[0],color=LOT_COLOR[k],lw=2,label=f"{k} · SPCD 실제 신호"),
        Patch(facecolor="#f5d98c",label="초기 승온 형태"),
        Patch(facecolor=COL["TBM"],label="하단 TBM: 계획 점검 수행"),
        Patch(facecolor=COL["CBM"],label="하단 CBM: 마모량 초과"),
        Patch(facecolor="#b04482",alpha=.4,label="극값 포함 CP 펄스"),
        Line2D([0],[0],color="#526d80",ls="--",label="CP-Z4 자체 안정 참고 중앙값")],ncol=3)
effects=pulse_effects.set_index("센서")
fig.text(.075,.126,f"CP 극값 {cp.PEAK_TIME:%H:%M:%S}: CBM 종료 {cbm.END_DT:%H:%M}보다 {peak_gap:.2f}분 뒤 / 펄스 폭 {cp.PULSE_WIDTH_MIN:.2f}분, 실제 {int(cp.RECORDED_SECONDS)}행.",
         fontsize=13,color="#17344d",fontweight="bold")
fig.text(.075,.082,f"전5분 대비 TC-Z4 {effects.loc['TC-Z4','펄스-전5분']:+.3f}℃ / OP-Z4 {effects.loc['OP-Z4','펄스-전5분']:+.3f}%p. TBM {tbm_duration:.0f}분·CBM {cbm_duration:.0f}분, 공유 {shared_work:.0f}분 → 겹침 합집합 {work_union:.0f}분.",
         fontsize=12,color="#526d80")
fig.text(.075,.035,"입력 결정: TC·OP 기록과 작업 맥락을 따로 보존한다. CP는 정의·단위가 미확인이라 보조 특징으로 두며 정비 효과·센서 고장 라벨로 해석하지 않는다.",
         fontsize=12.5,color="#17344d",fontweight="bold")
finish(fig,"01_구간근거/03_정비시각_CP_대조.png")
table(pulse_effects)
table(tasks[["WO_NO","DETC_DT","STRT_DT","END_DT","MNT_TYPE","REMARK"]])
note(f"**그래프 결과:** CP 펄스는 CBM 종료 {peak_gap:.2f}분 뒤이며 작업과 겹치지 않는다. 같은 펄스에서 TC-Z4와 OP-Z4 중앙값은 각각 {effects.loc['TC-Z4','펄스-전5분']:+.3f}℃·{effects.loc['OP-Z4','펄스-전5분']:+.3f}%p 달랐다. 이를 인과 반응이나 공학적 이상으로 확정할 근거는 없다. CP 스케일·계측 정의를 확인하기 전 주 입력에서 분리한다.")


### 감지 전후 신호와 정비 후 관측 공백

**목적:** 감지 양쪽의 실제 TC·OP 차이와 작업·교체 WO 완료 시각을 대조한다. 정비 이후 회복 효과를 확인할 온도 기록이 있는지 점검한다.

In [ ]:

note("**그림 목적 — 감지 전후 신호와 정비 후 관측 여부:** 240133 전체에서 감지 시각을 표시하고 양쪽 20분의 TC-Z3·OP-Z3 원본을 대조한다. 실제 작업·부품교체 WO 완료를 같은 날짜의 별도 시간축에 놓아 정비 후 효과를 관측할 자료가 있는지 확인한다.")
k=240133;meta=lot_meta.loc[k]
task=cal_work.loc[cal_work.DETC_DT.between(meta.START,meta.END)].iloc[0]
detect_min=(task.DETC_DT-meta.START).total_seconds()/60
work_gap=(task.STRT_DT-meta.END).total_seconds()/60
after_work_rows=int(temp.MEAS_DT.ge(task.END_DT).sum())
detect_values=[]
for s in ["TC-Z3","OP-Z3"]:
    g=raw[k]
    pre=g.loc[g.ELAPSED.ge(detect_min-20)&g.ELAPSED.lt(detect_min),s].dropna()
    post=g.loc[g.ELAPSED.ge(detect_min)&g.ELAPSED.lt(detect_min+20),s].dropna()
    detect_values.append({"LOT":k,"센서":s,"감지 전20분 중앙값":pre.median(),"감지 후20분 중앙값":post.median(),
                          "후-전":post.median()-pre.median(),"전 유효 행":len(pre),"후 유효 행":len(post)})
detect_values=pd.DataFrame(detect_values)
part_records=cal_parts.loc[cal_parts.WO_NO.eq(task.WO_NO)]
part_text=" · ".join(f"{z.PART_CD} {z.QTY}개" for z in part_records.itertuples())
part_color="#9a6324"
fig=plt.figure(figsize=(24,14))
gs=fig.add_gridspec(3,2,left=.075,right=.97,top=.78,bottom=.23,
                   hspace=.80,wspace=.24,height_ratios=[1,1,.58])
axs=np.array([[fig.add_subplot(gs[i,j]) for j in range(2)] for i in range(2)])
for i in range(2):
    for j,s in enumerate(["TC-Z3","OP-Z3"]):
        ax=axs[i,j]
        evidence_trace(ax,k,s,detect_min-20 if i else 0,detect_min+20 if i else None,detail=bool(i))
        ax.axvline(detect_min,color="#34495e",ls=":",lw=1.5)
        ax.axvspan(detect_min-20,detect_min,color="#0072b2",alpha=.1,lw=0)
        ax.axvspan(detect_min,detect_min+20,color="#d55e00",alpha=.1,lw=0)
        ax.set_xlim(detect_min-20,detect_min+20) if i else ax.set_xlim(0,meta.MINUTES)
        ax.set_title(f"{'감지 전후 원본' if i else '전체 관측'} · 가열·균열 [{s}]",loc="left",pad=12)
ax=fig.add_subplot(gs[2,:])
ax.plot([meta.START,meta.END],[0,0],color=LOT_COLOR[k],lw=7,solid_capstyle="butt")
ax.scatter([task.DETC_DT],[1],marker="v",color="#34495e",s=65)
ax.plot([task.STRT_DT,task.END_DT],[2,2],color=COL[task.MNT_TYPE],lw=9,solid_capstyle="butt")
ax.scatter([task.END_DT],[3],marker="D",color=part_color,s=65)
ax.set_yticks([0,1,2,3],["온도 관측","감지 [DETC_DT]",f"{task.MNT_TYPE} 작업","교체 WO 완료"])
for tick,color in zip(ax.get_yticklabels(),[LOT_COLOR[k],"#34495e",COL[task.MNT_TYPE],part_color]):
    tick.set_color(color)
ax.invert_yaxis()
ax.set_ylim(3.5,-.5)
ax.set_xlim(meta.START,task.END_DT+pd.Timedelta(minutes=10))
ax.axvspan(meta.END,task.END_DT+pd.Timedelta(minutes=10),color="#e8eff3",alpha=.8,lw=0,zorder=0)
ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.set_title(f"실제 시각 대조 · {task.WO_NO} · {task.REMARK}",loc="left",pad=12)
style(ax,"12월 28일 기록 시각[시:분]")
header(fig,"감지 시각·신호 대조",
       f"{k} SPCE · {meta.START:%H:%M}–{meta.END:%H:%M:%S} 관측; 감지와 작업 착수·완료를 구분",
       [Line2D([0],[0],color=LOT_COLOR[k],lw=2,label=f"{k} · SPCE 실제 신호"),
        Patch(facecolor="#0072b2",alpha=.2,label="감지 전20분"),
        Patch(facecolor="#d55e00",alpha=.2,label="감지 후20분"),
        Line2D([0],[0],color="#34495e",ls=":",label="정비 기록의 감지 시각"),
        Patch(facecolor="#e8eff3",label="온도 미관측"),
        Patch(facecolor=COL[task.MNT_TYPE],label=f"{task.MNT_TYPE}: {task.REMARK}"),
        Line2D([0],[0],color=part_color,marker="D",ls="None",label=f"{part_text}: 교체 WO 완료")],ncol=4)
dd=detect_values.set_index("센서")
fig.text(.075,.169,f"감지 {task.DETC_DT:%H:%M}: 전후 TC-Z3 중앙값 {dd.loc['TC-Z3','후-전']:+.3f}℃ / OP-Z3 {dd.loc['OP-Z3','후-전']:+.3f}%p.",
         fontsize=13,color="#17344d",fontweight="bold")
fig.text(.075,.122,f"작업 {task.STRT_DT:%H:%M}–{task.END_DT:%H:%M}: 관측 종료 {work_gap:.1f}분 뒤 착수, 이 작업 완료 후 제공 온도 {after_work_rows}행.",
         fontsize=12.4,color="#526d80")
fig.text(.075,.074,"입력 결정: 관측 중 TC·OP 특징은 유지한다. 감지·교체는 사후 맥락이며 회복 효과·고장 정답 라벨은 만들 수 없다.",
         fontsize=12.8,color="#17344d",fontweight="bold")
fig.text(.075,.032,"감지 전후 차이는 동시 관측 사실이다. 감지 시각을 실제 고장 발생 시각으로, 작업 착수를 회복 시작 시각으로 사용하지 않는다.",
         fontsize=11.7,color="#526d80")
finish(fig,"01_구간근거/04_감지시각_신호대조.png")
table(detect_values)
table(cal_parts.loc[cal_parts.WO_NO.eq(task.WO_NO),["WO_NO","PART_CD","PART_NM","QTY","CHG_RSN","END_DT"]])
note(f"**그래프 결과:** 감지 양쪽의 TC·OP 값과 표본 수는 확인했다. 실제 작업은 관측 종료 {work_gap:.1f}분 뒤 시작하며, 작업 완료 후 온도 {after_work_rows}행이라 부품 교체 후 회복을 대조할 수 없다. 작업·감지 기록은 해석 맥락으로 보존하고 온라인 입력에서는 제외한다.")


### 실제 근거 → 입력 결정

**목적:** 각 대조의 수치와 시간 근거에서 현재 센서값·과거 특징·사후 맥락·미확정 라벨의 처리 규칙을 도출한다.

In [ ]:

# 마지막에 실제 수치가 어떤 입력 규칙으로 이어졌는지 한 장으로 연결한다.
g117=raw[240117]
initial117=g117.loc[g117.ELAPSED.lt(5),HEAT].median()
prework_rise=(g117.loc[g117.ELAPSED.between(25,30),HEAT].median()-initial117).min()
first_work117=(cal_work.loc[cal_work.STRT_DT.lt(lot_meta.loc[240117,"END"])&
                            cal_work.END_DT.gt(lot_meta.loc[240117,"START"]),"STRT_DT"].min()-
               lot_meta.loc[240117,"START"]).total_seconds()/60
note("**그림 목적 — 실제 근거와 입력 결정:** 위 네 사례에서 무엇을 비교했고 어떤 값을 확인했는지, 현재 입력·사후 검토·미확정 라벨에 어떤 영향을 주는지 요약한다. 사례는 입력 처리 규칙을 설명하며 전체 LOT의 재계산·센서별 마스크·시간 누출 검증은 뒤 절에서 확인한다.")
decisions=[
    ["240044 · SPHC\nTC·OP 기록",
     f"초기 각 분 TC 유효 초 {valid_tc}\nOP 유효 초 {valid_op}\n각 분 기대 60초; 입력 기준 ≥54초",
     "시각 누락과 OP 값 NaN은 다름\n같은 시간 TC는 입력 조건 충족\nOP 0·1분은 미달",
     "TC 현재값 유지\nOP 0·1분 현재값 NA\n센서별 마스크 적용"],
    ["240061 · SPHC\n동일 강종 대조",
     f"TC-Z3 {a}–{b}분, 대조 2 LOT\n실제 비교 {int(e.RAW_COMPARE_SECONDS)}초 모두 |차이|≥3℃\n최대 {e.RAW_MAX_ABS_DELTA_C:.3f}℃ / OP=0 {op_zero_rows}/{len(candidate_records)}행",
     "온도 차이와 출력 0이 함께 관측됨\n제어 반응·설정·원인은 추가 확인\n대조에는 후속 시기 LOT도 포함",
     "실제 TC·OP 값과 과거 특징 유지\n같은 강종 차이는 사후 검토에 보존\n후속 LOT 차이는 온라인 입력 제외"],
    ["240117 · SPCD\n승온·작업",
     f"첫 작업 착수 {first_work117:.0f}분\n그 전 25–30분에 4 TC 모두\n최소 +{prework_rise:.2f}℃ 상승",
     "상승이 기록된 작업보다 먼저 나타남\n승온 형태와 작업은 다른 근거",
     "현재 TC·OP 값은 유지\n승온 형태·작업 기록을 독립 보존\n사후 상태·미래 정비는 온라인 제외"],
    ["240117 · SPCD\nCP·작업 종료",
     f"CP 극값 {cp.PEAK_VALUE:.6f}, 참고와 {cp.DELTA:+.6f}\nCBM 종료 {peak_gap:.2f}분 뒤\n동시 TC {effects.loc['TC-Z4','펄스-전5분']:+.3f}℃ / OP {effects.loc['OP-Z4','펄스-전5분']:+.3f}%p",
     "CP 펄스는 실제 작업과 겹치지 않음\n작업의 이전 영향은 미확정\nCP 정의·단위 미확인",
     "TC·OP와 CP를 별도로 보존\nCP는 보조 특징으로 분리\n정비 효과·고장 라벨로 사용 불가"],
    ["240133 · SPCE\n감지·교체 효과",
     f"감지 전후 TC-Z3 {dd.loc['TC-Z3','후-전']:+.3f}℃\n작업 착수는 관측 종료 {work_gap:.1f}분 뒤\n작업 완료 후 온도 {after_work_rows}행",
     "감지 양쪽의 신호 비교는 가능\n정비 이후 효과 대조 자료는 없음",
     "관측 중 TC·OP 특징 유지\n감지·WO·부품은 사후 맥락\n회복 효과·실제 고장 라벨 NA"]]
input_decisions=pd.DataFrame(decisions,columns=["대조 대상","실제 근거","확인한 사실","입력·검토 처리"])
table(input_decisions)
from matplotlib.patches import Rectangle
fig=plt.figure(figsize=(25,11))
header(fig,"근거·입력 결정 대조","플래그 개수 대신 실제 유효 초·온도 차이·정비 시각·관측 공백에서 모델링 준비의 처리 규칙으로 연결",
       [Patch(facecolor="#e5f1fa",label="실제 대조 수치"),
        Patch(facecolor="#e6f4ed",label="확인한 사실"),
        Patch(facecolor="#f1eafa",label="입력·검토 처리")],ncol=3)
xs=[.035,.185,.495,.73];widths=[.15,.31,.235,.235]
top=.79;height=.104
for x,width,name in zip(xs,widths,["대조 대상","실제 근거","무엇을 알았는가","어떤 값을 남기는가"]):
    fig.add_artist(Rectangle((x,top),width,.043,transform=fig.transFigure,facecolor="#17344d",edgecolor="white"))
    fig.text(x+.009,top+.012,name,color="white",fontsize=14,fontweight="bold",va="bottom")
for i,row in enumerate(decisions):
    y=top-(i+1)*height
    for j,(x,width,value) in enumerate(zip(xs,widths,row)):
        color="#f7fafc" if j==0 else ["#e5f1fa","#e6f4ed","#f1eafa"][j-1]
        fig.add_artist(Rectangle((x,y),width,height,transform=fig.transFigure,facecolor=color,edgecolor="white",lw=1.3))
        fig.text(x+.009,y+height-.018,value,va="top",fontsize=12.6 if j else 13.5,
                 linespacing=1.6,color="#273c4a",fontweight="bold" if j==0 else "normal")
fig.text(.035,.190,"결론: 실제 유효 센서값·과거 특징은 입력에 남기고, 사후 비교·정비·감지·CP 의미는 별도로 관리한다.",
         fontsize=14,color="#17344d",fontweight="bold")
fig.text(.035,.139,"현재 자료는 입력 준비와 검토 근거를 제공한다. 실제 고장·정상·더미 정답과 정비 효과는 추가 기록이 필요하다.",
         fontsize=12.5,color="#526d80")
fig.text(.035,.081,"1분 기준: 온라인 입력은 닫힌 60초의 유효 기록 ≥54초. 비교 초는 동일 시각에 대상과 대조가 모두 유효한 초만 세며 보간하지 않는다.",
         fontsize=11.8,color="#526d80")
finish(fig,"01_구간근거/05_근거_입력판단_대조.png")
note("**그래프 결과:** 결측은 해당 센서·분의 현재값에, 같은 강종 차이는 사후 검토에, 실제 작업·감지는 사건 맥락에 각각 반영한다. TC·OP의 현재값과 과거 특징을 사용하는 계산은 아래 4절에서 수행한다. 사후 대조·운전 형태·미래 사건을 온라인 입력에서 제외하는 규칙과 실제 마스크는 7절에서 검증한다.")


## 3. 모델 목표와 실제 라벨

**목적:** 관측 사실·사후 후보를 실제 고장 정답으로 오해하지 않는다. FAULT_LABEL·DUMMY_LABEL·NORMAL_LABEL은 nullable boolean의 NA로 남긴다. 정상 후보로 보이는 안정 참고도 아직 검증된 정상은 아니다.

현재 준비 가능한 목표는 **운전 조건을 확인하며 미설명 신호 변화를 검토하는 이상 탐색**과 **센서 값·변동의 시간 예측**이다. 작업 발생 예측은 별도 업무 목표이며 BM/CBM/TBM 작업이 실제 고장과 같지 않다. 고장 분류·RUL·정비 효과 평가에는 실제 고장·정답 시각·사용 시간·정비 후 관측이 필요하다.

In [ ]:
label_summary=minute_context.groupby("LOT")[["FAULT_LABEL","DUMMY_LABEL","NORMAL_LABEL"]].count()
table(label_summary)
assert label_summary.to_numpy().sum()==0
table(pd.DataFrame([
    {"값":"TEMP_REVIEW / CP_REVIEW","의미":"탐색 규칙으로 만든 검토 위치","사용":"사후 검증·검토 우선순위","정답 라벨":"아님"},
    {"값":"CAL_WORK / CAL_DETECT / PART_WO_COMPLETED","의미":"기록된 보전 사건 맥락","사용":"사후 층화·시각화","정답 라벨":"고장 여부 미확정"},
    {"값":"REVIEW_STATE","의미":"전체 관측으로 나눈 형태","사용":"사후 비교·학습 기준 검토","정답 라벨":"실제 운전 코드 아님"},
    {"값":"센서별 VALID_RATIO","의미":"원본 기대 초 대비 유효율","사용":"품질 마스크","정답 라벨":"센서 고장 증거 아님"},
    {"값":"FAULT/DUMMY/NORMAL_LABEL","의미":"실제 확인 자료 없음","사용":"NA로 보존","정답 라벨":"미제공"}]))
note("**결과:** 실제 정답이 있는 분당창은 0개다. 고장 예측 정확도·재현율이나 RUL 오차를 계산할 근거가 없다. 03의 가설 지지·기각·보류는 분석 결과이고 학습 정답으로 자동 변환하지 않는다.")

## 4. 예측 시점에 알 수 있는 특징

**목적:** 현재 1분창을 닫은 시각을 PREDICTION_TIME으로 두고, 현재 원본 집계와 이전 분들만으로 특징을 계산한다. 온라인 품질의 분모는 항상 60초이며 마지막 관측창도 1분 종료까지 기다린다. 사후 표의 마지막 부분 분 품질과 구분한다.

센서 시각과 강종 이벤트 시각에 즉시 이용 가능하다고 가정한 준비 표다. 실제 수집 지연·기록 등록 시각은 제공되지 않아 운영 적용 전에 확인해야 한다. 다른 설비의 WO·REMARK·교체 사유·미래 감지·전체 LOT 통계·평가 LOT 기준값은 입력하지 않는다. 중심 이동 평균·미래 경계로 추정한 상태·사후 후보도 입력하지 않는다.

TC-Z9·냉각은 이전 60분·300분 범위의 변동폭을 추가하며 완결 주기·위상을 온라인 입력으로 재사용하지 않는다. 긴 창이 채워지지 않은 값은 NA로 남긴다.

In [ ]:
# 입력은 관측 시각에 알 수 있는 센서 기록·강종 이벤트로 제한한다.
# 작업 REMARK·교체·사후 상태·후보 플래그·전체 LOT 통계는 온라인 특징에 넣지 않는다.
def features_for_lot(g,lot,grade,start):
    g=g.sort_values("MEAS_DT").copy()
    g["MINUTE"]=np.floor((g.MEAS_DT-start).dt.total_seconds()/60).astype(int)
    grid=pd.RangeIndex(g.MINUTE.max()+1,name="MINUTE")
    grouped=g.groupby("MINUTE")[SIGNALS]
    agg=grouped.agg(["median","min","max","count"]).reindex(grid)
    iqr=(grouped.quantile(.75)-grouped.quantile(.25)).reindex(grid)
    result={"LOT":np.full(len(grid),lot),"GRADE":np.full(len(grid),grade),"MINUTE":grid.to_numpy(),"PREDICTION_TIME":start+pd.to_timedelta(grid+1,unit="min")}
    last=g.groupby("MINUTE").MEAS_DT.max().reindex(grid)
    result["MAX_SOURCE_TIME"]=last.to_numpy()
    result["GRADE_EVENT_TIME"]=start
    for s in SIGNALS:
        # 온라인에서는 닫힌 60초 창을 기다린다. 마지막 짧은 관측도 60초를 분모로 사용한다.
        q=agg[(s,"count")].fillna(0)/60
        y=agg[(s,"median")].where(q.ge(.9))
        result[s+"_CURRENT"]=y.to_numpy()
        result[s+"_IQR"]=iqr[s].where(q.ge(.9)).to_numpy()
        result[s+"_VALID_RATIO_60"]=q.to_numpy()
        result[s+"_DELTA1"]=y.diff().to_numpy()
        past=y.shift(1)
        result[s+"_PAST5_MEDIAN"]=past.rolling(5,min_periods=3).median().to_numpy()
        result[s+"_PAST5_STD"]=past.rolling(5,min_periods=3).std().to_numpy()
        result[s+"_PAST60_Q95_Q05"]=(past.rolling(60,min_periods=30).quantile(.95)-past.rolling(60,min_periods=30).quantile(.05)).to_numpy()
        if s in ["TC-Z9","TC-OUT1","TC-OUT2"]:
            result[s+"_PAST300_Q95_Q05"]=(past.rolling(300,min_periods=180).quantile(.95)-past.rolling(300,min_periods=180).quantile(.05)).to_numpy()
    return pd.DataFrame(result)

feature_frames=[features_for_lot(raw[int(r.LOT_NO)],int(r.LOT_NO),r.GRADE,r.START) for r in lots.itertuples()]
features=pd.concat(feature_frames,ignore_index=True).sort_values(["PREDICTION_TIME","LOT"]).reset_index(drop=True)
feature_columns=[c for c in features.columns if c not in ["LOT","GRADE","MINUTE","PREDICTION_TIME","MAX_SOURCE_TIME","GRADE_EVENT_TIME"]]
assert features.MAX_SOURCE_TIME.dropna().lt(features.loc[features.MAX_SOURCE_TIME.notna(),"PREDICTION_TIME"]).all()
assert features.GRADE_EVENT_TIME.le(features.PREDICTION_TIME).all()


In [ ]:
k=240061
d=features.loc[features.LOT.eq(k)].set_index("MINUTE")
fig,axs=plt.subplots(2,2,figsize=(24,11))
fig.subplots_adjust(left=.10,right=.97,top=.77,bottom=.12,wspace=.28,hspace=.68)
for j,s in enumerate(["TC-Z3","OP-Z3"]):
    axs[0,j].plot(d.index,d[s+"_CURRENT"],color=LOT_COLOR[k],lw=1.8)
    axs[0,j].plot(d.index,d[s+"_PAST5_MEDIAN"],color="#455a64",ls="--",lw=1.5)
    axs[0,j].set_title(f"{PROCESS[s]} [{s}] · 현재·과거",loc="left",pad=12)
    axs[0,j].set_ylabel(unit(s),labelpad=10)
    axs[1,j].plot(d.index,d[s+"_VALID_RATIO_60"]*100,color="#0072b2",lw=1.5)
    axs[1,j].set_title(f"{PROCESS[s]} [{s}] · 닫힌 60초창 품질",loc="left",pad=12)
    axs[1,j].set_ylabel("1초 기대 기록 대비 유효율[%]",labelpad=10)
    axs[1,j].set_ylim(0,105)
    axs[1,j].axhline(90,color="#c83745",ls=":",lw=1)
for ax in axs.flat:
    style(ax,"해당 1분창 시작의 경과시간[분]")
header(fig,"현재·과거 특징 대조","240061 · 현재 값은 1분창을 닫은 뒤 사용; 점선은 현재를 제외한 이전 5분",
       [Line2D([0],[0],color=LOT_COLOR[k],label="현재 닫힌 분 중앙값"),
        Line2D([0],[0],color="#455a64",ls="--",label="이전 5분 중앙값"),
        Line2D([0],[0],color="#0072b2",label="60초 기준 센서 품질"),
        Line2D([0],[0],color="#c83745",ls=":",label="현재 특징 사용 조건 90%")])
finish(fig,"02_특징준비/01_현재_과거_품질.png")
for signal,suffix in [("TC-Z3","℃"),("OP-Z3","%p")]:
    gap=d[signal+"_CURRENT"]-d[signal+"_PAST5_MEDIAN"]
    first=d[signal+"_PAST5_MEDIAN"].first_valid_index()
    note(f"**그림 결과 — {signal}:** 현재−이전 5분 중앙값 범위는 {gap.min():+.2f}–{gap.max():+.2f}{suffix}, 과거 특징의 첫 유효 시각은 경과 {first:.0f}분이다. 60초 기준 품질 90% 미만 창 {int(d[signal+'_VALID_RATIO_60'].lt(.9).sum())}개의 현재 값은 NA다. 과거 곡선의 반응이 뒤따르는 모습과 품질에 따른 빈 구간을 함께 확인할 수 있다.")
note("**모델링에 쓰는 판단:** 현재와 과거의 차이를 승온·변동 특징으로 준비할 수 있다. 이 그래프만으로 고장 구분 성능을 확인한 것은 아니며 실제 고장 라벨과 운전 조건을 확보한 뒤 시간 순서 평가가 필요하다.")
table(features.loc[features.LOT.eq(k),["LOT","MINUTE","PREDICTION_TIME","MAX_SOURCE_TIME","TC-Z3_CURRENT","TC-Z3_PAST5_MEDIAN","TC-Z3_VALID_RATIO_60"]].head(10))
dictionary=pd.DataFrame([
    {"특징":"CURRENT","계산":"닫힌 1분의 원본 중앙값","단위":"TC ℃ / OP % / CP 원본 기록값","조건":"해당 센서 60초 유효율 ≥90%"},
    {"특징":"IQR","계산":"현재 분 Q75−Q25","단위":"TC ℃ / OP %p / CP 기록값 차이","조건":"현재 품질 조건"},
    {"특징":"DELTA1","계산":"현재−바로 이전 분","단위":"TC ℃ / OP %p / CP 기록값 차이","조건":"두 분 유효"},
    {"특징":"PAST5_MEDIAN·STD","계산":"현재 제외 이전 5분","단위":"각 센서 원본 단위","조건":"유효 분 3개 이상"},
    {"특징":"PAST60_Q95_Q05","계산":"현재 제외 이전 60분 범위","단위":"TC ℃ / OP %p / CP 기록값 차이","조건":"유효 분 30개 이상"},
    {"특징":"PAST300_Q95_Q05","계산":"TC-Z9·냉각의 이전 300분 범위","단위":"온도 변동폭[℃]","조건":"유효 분 180개 이상"},
    {"특징":"VALID_RATIO_60","계산":"유효 기록 개수/60","단위":"비율[0–1]","조건":"기록이 없는 분은 0"}])
table(dictionary)
note(f"**결과:** 센서 특징 {len(feature_columns)}개·분당 행 {len(features):,}개를 노트북 변수로 준비했다. 초기 이동 특징과 긴 주기 창은 표본 부족 시 NA다. CP-Z4M은 정의 미확인 원본 스케일을 유지했고 전체 자료 중앙값으로 % 정규화하지 않았다.")

## 5. LOT 단위 시간 순서 분할

**목적:** 미래 LOT과 같은 LOT의 일부가 학습·평가 양쪽에 들어가지 않게 한다. 앞 8 LOT(1–7월)을 학습 준비, 뒤 6 LOT(9–12월)을 후속 평가 준비로 둔다. 분할은 관측 시간 순서로 고정하며 신호 후보나 정비 정답을 보고 고르지 않는다.

이 후속 자료는 이미 탐색 과정에서 보았으므로 독립 미공개 성능 평가 자료가 아니다. 모델·문턱 선택 후 성능을 주장하려면 이후의 새 기간과 확인된 라벨이 필요하다. 작은 LOT 수에서는 초 단위 표본 수를 독립 사례 수로 계산하지 않는다.

In [ ]:
split_lots=lots[["LOT_NO","GRADE","START","END"]].rename(columns={"LOT_NO":"LOT"}).copy()
split_lots["SPLIT"]=np.where(np.arange(len(split_lots))<8,"학습 준비","후속 평가 준비")
split_map=split_lots.set_index("LOT").SPLIT
features["SPLIT"]=features.LOT.map(split_map)
train_ids=set(split_lots.loc[split_lots.SPLIT.eq("학습 준비"),"LOT"])
eval_ids=set(split_lots.loc[split_lots.SPLIT.eq("후속 평가 준비"),"LOT"])
assert train_ids.isdisjoint(eval_ids) and train_ids|eval_ids==set(lots.LOT_NO)
baseline_available_at=features.loc[features.SPLIT.eq("학습 준비"),"PREDICTION_TIME"].max()
assert baseline_available_at<features.loc[features.SPLIT.eq("후속 평가 준비"),"PREDICTION_TIME"].min()
fig,ax=plt.subplots(figsize=(24,10))
fig.subplots_adjust(left=.16,right=.97,top=.78,bottom=.14)
split_color={"학습 준비":"#0072b2","후속 평가 준비":"#d55e00"}
for i,r in enumerate(split_lots.itertuples()):
    ax.plot([r.START,r.END],[i,i],color=split_color[r.SPLIT],lw=5,solid_capstyle="butt")
    ax.scatter(r.START,i,color=split_color[r.SPLIT],s=28)
ax.axvline(baseline_available_at,color="#455a64",ls="--",lw=1.2)
ax.set_yticks(range(len(split_lots)),[f"{r.LOT} · {r.GRADE}" for r in split_lots.itertuples()])
ax.invert_yaxis();ax.set_xlim(YEAR_START,YEAR_END)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2));ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y/%m"))
ax.set_ylabel("전체 LOT[관측 시작순]",labelpad=12);style(ax,"관측 시각[년/월]")
header(fig,"학습·후속 평가 시간 분할","한 LOT은 하나의 분할에만 포함; 학습 기준 생성 시각이 모든 후속 예측 시각보다 앞섬",
       [Line2D([0],[0],color=col,lw=4,label=label) for label,col in split_color.items()]+[
        Line2D([0],[0],color="#455a64",ls="--",label="학습 자료 종료·기준 이용 가능 시각")])
finish(fig,"03_평가분할/01_LOT_시간분할.png")
table(split_lots)
split_grade=split_lots.groupby(["GRADE","SPLIT"]).size().unstack(fill_value=0).reindex(columns=["학습 준비","후속 평가 준비"],fill_value=0)
table(split_grade)
note(f"**결과:** 학습 {len(train_ids)} LOT·후속 {len(eval_ids)} LOT, 경계 {baseline_available_at:%Y/%m/%d %H:%M:%S}다. SPCD는 학습에 없어 새로운 강종이며, SPCE·SPCEN·SPFH590은 학습 대조 LOT이 1개뿐이다. 분할 이후 학습 기준값을 평가 자료로 보완하지 않는다.")

## 6. 학습 전용 기준값과 후속 활용 가능 범위

**목적:** 기준값도 학습 기간 안에서만 만든다. 학습의 사후 안정 참고·센서 품질 조건을 만족하는 각 LOT의 중앙값을 먼저 계산한 뒤 강종별 LOT 중앙값을 사용하여 긴 LOT 한 개가 기준을 지배하지 않게 한다. 실제 정상 기준으로 인증된 값은 아니다.

강종·센서의 대조 LOT이 2개 미만이면 기준을 미지원으로 표시하며 다른 강종 평균이나 후속 LOT으로 채우지 않는다. 이 기준은 학습이 끝난 뒤의 후속 예측에만 적용한다. 과거 학습 시점의 온라인 특징에 미래의 학습 종료 기준을 붙이지 않는다.

In [ ]:
def fit_training_reference(data,ids):
    rows=[]
    for k in sorted(ids):
        r=lot_meta.loc[k];f=data.loc[data.LOT.eq(k)].set_index("MINUTE")
        st=states[k].reindex(f.index)
        for s in SIGNALS:
            y=f[s+"_CURRENT"].where(st.eq("안정 참고")).dropna()
            rows.append({"LOT":k,"GRADE":r.GRADE,"SENSOR":s,"N_VALID_MINUTES":len(y),
                         "LOT_MEDIAN":y.median() if len(y)>=30 else np.nan})
    levels=pd.DataFrame(rows)
    ref=levels.groupby(["GRADE","SENSOR"],as_index=False).agg(REF_MEDIAN=("LOT_MEDIAN","median"),N_REF_LOTS=("LOT_MEDIAN","count"))
    ref["SUPPORTED"]=ref.N_REF_LOTS.ge(2)
    ref["REF_MEDIAN"]=ref.REF_MEDIAN.where(ref.SUPPORTED)
    ref["AVAILABLE_AT"]=data.loc[data.LOT.isin(ids),"PREDICTION_TIME"].max()
    return levels,ref
train_levels,baseline=fit_training_reference(features,train_ids)
evaluation_features=features.loc[features.SPLIT.eq("후속 평가 준비")].copy()
ref_index=baseline.set_index(["GRADE","SENSOR"])
for s in SIGNALS:
    refs=baseline.loc[baseline.SENSOR.eq(s)].set_index("GRADE")
    evaluation_features[s+"_N_REF_LOTS"]=evaluation_features.GRADE.map(refs.N_REF_LOTS).fillna(0).astype(int)
    evaluation_features[s+"_TRAIN_REFERENCE"]=evaluation_features.GRADE.map(refs.REF_MEDIAN)
    evaluation_features[s+"_TRAIN_DELTA"]=evaluation_features[s+"_CURRENT"]-evaluation_features[s+"_TRAIN_REFERENCE"]
evaluation_features["BASELINE_AVAILABLE_AT"]=baseline_available_at
assert evaluation_features.BASELINE_AVAILABLE_AT.le(evaluation_features.PREDICTION_TIME).all()
support=split_grade.copy()
support["2 LOT 이상 학습 기준"]=support["학습 준비"].ge(2)
fig,ax=plt.subplots(figsize=(20,9))
fig.subplots_adjust(left=.16,right=.96,top=.76,bottom=.15)
x=np.arange(len(support));width=.32
ax.bar(x-width/2,support["학습 준비"],width,color="#0072b2")
ax.bar(x+width/2,support["후속 평가 준비"],width,color="#d55e00")
ax.set_xticks(x,support.index);ax.set_ylabel("독립 LOT[개]",labelpad=10)
ax.yaxis.set_major_locator(MaxNLocator(integer=True));style(ax,"강종[원본 조업 이벤트]")
header(fig,"강종별 학습·평가 대조","2개 미만 학습 LOT은 강종별 기준 부족; 모든 강종은 원본·메타데이터에 보존",
       [Patch(facecolor="#0072b2",label="학습 준비 LOT"),Patch(facecolor="#d55e00",label="후속 평가 준비 LOT")])
finish(fig,"03_평가분할/02_강종별_참고부족.png")
table(support);table(baseline.loc[baseline.SENSOR.isin(["TC-Z3","OP-Z3","CP-Z4","CP-Z4M"])])
supported=evaluation_features.loc[evaluation_features["TC-Z3_N_REF_LOTS"].ge(2),"LOT"].unique().tolist()
note(f"**결과:** 2 LOT 이상 학습 기준이 있는 후속 LOT은 {supported}다. 다른 LOT의 차이는 NA로 남긴다. 후속 자료를 기준에 섞어 지원 범위를 늘리는 것은 누출이다. 학습 참고값이 있는 LOT도 실제 정상·고장 라벨은 없다.")

## 7. 미래 정보 누출과 연결 일관성 검증

**목적:** 특징 계산이 미래 값·미래 행수에 영향을 받지 않는지 실제로 검사한다. 240061의 60분과 240069의 360분을 경계로 이후 모든 신호를 크게 바꾸고, 별도로 미래 행을 모두 제거하여 경계 이전 특징이 같은지 확인한다. 360분 사례는 300분 특징이 실제로 계산된 구간도 검증한다. 경계를 포함한 분은 이미 닫힌 창만 비교한다.

시간 구간·특징 키, 원본 NaN·시각 누락의 합, 작업·승온 중복 보존, 센서 마스크와 미확정 라벨을 검사한다. 사후 기준·상태가 온라인 입력에 포함되지 않았는지도 확인한다.

In [ ]:
checks=[]
def checked(name,condition,detail):
    assert bool(condition),name
    checks.append({"검증":name,"결과":"통과","근거":detail})
plot_case=None
for k,cut_min in [(240061,60),(240069,360)]:
    r=lot_meta.loc[k];cut=r.START+pd.Timedelta(minutes=cut_min)
    original=features_for_lot(raw[k],k,r.GRADE,r.START)
    changed=raw[k].copy();changed.loc[changed.MEAS_DT.ge(cut),SIGNALS]=999999.0
    changed_features=features_for_lot(changed,k,r.GRADE,r.START)
    deleted=features_for_lot(raw[k].loc[raw[k].MEAS_DT.lt(cut)],k,r.GRADE,r.START)
    cols=[c for c in original if c not in ["SPLIT"]]
    left=original.loc[original.PREDICTION_TIME.le(cut),cols].reset_index(drop=True)
    pd.testing.assert_frame_equal(left,changed_features.loc[changed_features.PREDICTION_TIME.le(cut),cols].reset_index(drop=True))
    pd.testing.assert_frame_equal(left,deleted.loc[deleted.PREDICTION_TIME.le(cut),cols].reset_index(drop=True))
    checked(f"{k} 미래 값 변조 불변",True,f"{len(left)}개 닫힌 분의 모든 특징 동일")
    checked(f"{k} 미래 행 삭제 불변",True,"관측 종료·분모·격자 변경이 과거 특징에 역류하지 않음")
    if plot_case is None:
        plot_case=(original,changed_features,cut)
original,changed_features,cut=plot_case
checked("LOT 분할 중복 없음",train_ids.isdisjoint(eval_ids),"8 학습 LOT / 6 후속 LOT")
checked("전체 시간 순서 분할",baseline_available_at<evaluation_features.PREDICTION_TIME.min(),"학습 기준 생성이 후속 예측보다 앞섬")
checked("현재 특징 원본 시각",features.MAX_SOURCE_TIME.dropna().lt(features.loc[features.MAX_SOURCE_TIME.notna(),"PREDICTION_TIME"]).all(),"현재 분 기록은 예측 시각보다 이전")
checked("강종 이벤트 시각",features.GRADE_EVENT_TIME.le(features.PREDICTION_TIME).all(),"이벤트 발생 시각 이용; 실제 수집 지연은 별도 확인 필요")
checked("구간·특징 키 일치",set(zip(minute_context.LOT,minute_context.MINUTE))==set(zip(features.LOT,features.MINUTE)),"LOT·MINUTE 전수 키 일치")
for row in lots.itertuples():
    k=int(row.LOT_NO)
    duration=minute_context.loc[minute_context.LOT.eq(k),"REVIEW_INTERVAL_SECONDS"].sum()
    checked(f"{k} 구간 합",np.isclose(duration,(row.END-row.START).total_seconds()+1),"기대 1초 관측창 끝까지 중복 없이 분할")
checked("실제 라벨 NA",minute_context[["FAULT_LABEL","DUMMY_LABEL","NORMAL_LABEL"]].notna().sum().sum()==0,"정비·후보를 고장 정답으로 만들지 않음")
checked("승온·작업 중복 보존",((minute_context.LOT==240117)&(minute_context.REVIEW_STATE=="초기 승온")&minute_context.CAL_WORK).any(),"동일 시간 두 근거 보존")
for s in SIGNALS:
    low=features[s+"_VALID_RATIO_60"].lt(.9)
    checked(f"{s} 온라인 품질 마스크",features.loc[low,s+"_CURRENT"].isna().all(),"센서별 부족 현재값은 NA")
for name in ["REVIEW_STATE","TEMP_REVIEW","CP_REVIEW","CAL_WORK","CAL_DETECT","PART_WO_COMPLETED","FAULT_LABEL"]:
    checked(f"{name} 입력 제외",name not in features.columns,"사후 해석·정답·미래 사건은 특징에서 제외")
# 전체 입력의 후속 신호를 바꾼 뒤 같은 학습 함수의 출력이 변하지 않는지 확인한다.
changed_all=features.copy()
changed_all.loc[changed_all.LOT.isin(eval_ids),[s+"_CURRENT" for s in SIGNALS]]=999999.0
changed_levels,changed_baseline=fit_training_reference(changed_all,train_ids)
pd.testing.assert_frame_equal(train_levels,changed_levels)
pd.testing.assert_frame_equal(baseline,changed_baseline)
checked("학습 전용 기준값",set(train_levels.LOT).issubset(train_ids),"후속 특징·신호가 기준 계산 입력에 들어가지 않음")
validation_checks=pd.DataFrame(checks);table(validation_checks)
fig,axs=plt.subplots(1,2,figsize=(23,8))
fig.subplots_adjust(left=.10,right=.97,top=.77,bottom=.18,wspace=.27)
for ax,s in zip(axs,["TC-Z3_CURRENT","TC-Z3_PAST5_MEDIAN"]):
    a=original.loc[original.PREDICTION_TIME.le(cut)].set_index("MINUTE")[s]
    b=changed_features.loc[changed_features.PREDICTION_TIME.le(cut)].set_index("MINUTE")[s]
    ax.plot(a.index,a,color="#0072b2",lw=2.8)
    ax.plot(b.index,b,color="#d55e00",ls="--",lw=1.7)
    ax.set_title(f"가열·균열 [{s}]",loc="left",pad=12);ax.set_ylabel("온도[℃]",labelpad=10)
    style(ax,"관측 시작 후 경과시간[분]")
header(fig,"미래 변조 전후 특징 대조","경계 이후 모든 신호를 999999로 바꿔도 이미 닫힌 과거 60분의 특징은 동일",
       [Line2D([0],[0],color="#0072b2",label="원본 특징"),
        Line2D([0],[0],color="#d55e00",ls="--",label="미래 신호 변조 뒤 특징")])
finish(fig,"04_누출검증/01_미래변조_과거불변.png")
note(f"**검증 결과:** {len(validation_checks)}개 시간·키·품질·누출 조건이 통과했다. 과거 특징 불변은 실제 값 변조·행 삭제로 확인했다. 실제 등록 지연, 공학적 정상, 고장 정답의 타당성은 이 검사로 보증할 수 없어 추가 운영 자료가 필요하다.")

## 8. 최종 활용값과 모델링 전에 남기는 판단

**목적:** 실제 준비된 값·근거·용도를 한 표에 정리한다. 현재 변수들은 노트북 안에서 조회할 수 있고 추가 CSV·보조 스크립트·다른 노트북을 생성하지 않는다. 입력과 사후 해석용 자료를 연결하되 사후 결과가 입력에 역류하지 않게 한다.

In [ ]:
core_input_columns=[c for c in feature_columns if not c.startswith(("CP-Z4_","CP-Z4M_"))]
auxiliary_cp_columns=[c for c in feature_columns if c.startswith(("CP-Z4_","CP-Z4M_"))]
model_ready=features[["LOT","GRADE","MINUTE","PREDICTION_TIME","MAX_SOURCE_TIME","GRADE_EVENT_TIME","SPLIT"]+core_input_columns].copy()
review_ready=minute_context.merge(sensor_quality,on=["LOT","MINUTE"],validate="one_to_many")
usage=pd.DataFrame([
    {"변수":"lots / context","규모":f"{len(lots)} LOT","도출 근거":"원본 관측·정확 강종 시각·작업/용접 교집합","활용":"LOT 메타데이터·관측 범위"},
    {"변수":"temperature_validation","규모":f"{len(temperature_validation)} 센서 후보","도출 근거":"같은 강종·상태·원본 초·대조 2 LOT","활용":"원본 반복 검토; 고장 정답 아님"},
    {"변수":"cp_candidates","규모":f"{len(cp_candidates)} LOT","도출 근거":"자체 참고 대비 CP 원본 0.05 차이","활용":"단위·계측 의미 확인 전 별도 검토"},
    {"변수":"minute_context","규모":f"{len(minute_context):,} 분당 행","도출 근거":"사후 상태·작업/감지·후보 독립 플래그","활용":"검토·층화·이벤트 시각 대조"},
    {"변수":"sensor_quality / review_ready","규모":f"{len(sensor_quality):,} 센서·분 행","도출 근거":"센서별 기대 초 대비 유효율","활용":"센서별 품질·사후 근거 조회"},
    {"변수":"model_ready","규모":f"{len(model_ready):,}행 × {len(core_input_columns)} 센서 특징","도출 근거":"현재 닫힌 60초·이전 분만 사용","활용":"온도/출력 입력 후보; 학습 전 인코딩·마스킹 설계"},
    {"변수":"auxiliary_cp_columns","규모":f"{len(auxiliary_cp_columns)} 특징","도출 근거":"CP 원본 스케일 유지","활용":"정의·단위 확인 전 주 입력에서 분리"},
    {"변수":"baseline / evaluation_features","규모":f"{len(baseline)} 강종·센서 기준","도출 근거":"8 학습 LOT만; 대조 부족은 NA","활용":"학습 완료 뒤 후속 관측의 참고 차이"},
    {"변수":"split_lots / validation_checks","규모":f"{len(split_lots)} LOT·{len(validation_checks)} 검사","도출 근거":"LOT 단위 시간 순서·실제 미래 변조","활용":"누출 방지·후속 평가 지원 범위"}])
table(usage)
table(model_ready[["LOT","GRADE","MINUTE","PREDICTION_TIME","SPLIT","TC-Z3_CURRENT","OP-Z3_CURRENT","TC-Z9_PAST300_Q95_Q05"]].head(8))
note(f"**최종 결과:** 주 입력 후보 {len(core_input_columns)}개와 CP 보조 특징 {len(auxiliary_cp_columns)}개를 구분해 준비했다. 초반·장기 창 NA는 보간하거나 0으로 정상화하지 않는다. 온도 14 LOT 중 8개 학습·6개 후속으로 시간 분할했으며 실제 라벨은 모두 미확정이다.")
note("**남은 모델링 결정:** 실제 운전 조건·센서 사전 확인 → 입력의 단위·상태 처리 확정 → 학습 기간 안의 별도 시간 검증으로 문턱/모델 선택 → 이후 새 기간의 검토 라벨과 보전 업무 결과로 평가한다. 고장 예측·잔여수명·정비 효과의 성능 평가는 추가 정답 자료 확보 후 가능하다. 현재 수행 가능한 원본 전수 확인과 누출 검증은 위에서 완료했다.")
table(pd.DataFrame(figure_checks))
note(f"**시각화 확인:** {len(figure_checks)}개 그림의 글자 겹침·잘림·범례 위치 검사를 통과했다.")